# Simulate the gene expression in a population of cells

The code simulates gene expression based on a GRN (described by the interaction matrix) and expression of each gene is defined by parameters (each row in the parameter sheet) using the Gillespie algorithm.


In [4]:

import numpy as np
root = "/Users/laj2116/Desktop/TWINFER/grnInference/"

base_config = {
    'n_cells': 5000, #Number of cells before division (number of twin pairs)
    'steady_state_time': 2500, #The time used to run the initial cells till they reach steady-state [hours]
    'twin_sampling_duration': 48, #The time twin cells are simulated after division [hours]
    'twin_measurement_resolution': 1, #The time between each measurement of twin cells [hours]. For example, if twin_sampling_duration is 12 and twin_measurement_resolution is 1, the final dataframe will contain hourly measurements for 12 hours (0 is birth).
    "path_to_connectivity_matrix": f"{root}/data/example_input_data_for_simulation/example_interaction_matrix_3_gene_linear_cascade.txt", #path to the connectivity matrix specifying the GRN to simulate
    "param_csv": f"{root}/data/example_simulation_input/median_parameter.csv", #Path to the parameters for all genes and interaction terms
    "rows_to_use": [[0, 1, 2]], #Rows in the parameter's csv file for each gene - the length should be equal to number of genes in the system
    "output_folder": f"{root}/data/example_output_folder/", #Path to folder to store simulation 
    "log_file": f"{root}/data/example_output_folder/median_simulations.jsonl", #Path to the log file
    "type": "A_B",  # Name of the network used -- will be in the filename
    "number_of_parallel_parameters": 1, #Number of parameters to be run in parallel
    "number_of_cores_per_parameter": 8 #Number of cores to be used per parameter (number_of_parallel_parameters * number_of_cores_per_parameter = number of cores in your computer)
}


## Import functions from gillespie_script


In [11]:
from joblib import Parallel, delayed
from tqdm import tqdm
import os
from tqdm import tqdm
import sys

from gillespie_script import process_param_set
from numba import set_num_threads, get_num_threads

set_num_threads(base_config['number_of_cores_per_parameter'])
print("Threads Numba will use:", get_num_threads())

Threads Numba will use: 8


In [9]:
import os
from joblib import Parallel, delayed
from tqdm import tqdm

# Ensure output folder exists
os.makedirs(base_config['output_folder'], exist_ok=True)

rows_to_use = base_config['rows_to_use']

# Create 20 copies of each parameter set with unique label suffix
n_repeats = 9
arr = [x+12 for x in range(n_repeats)]

labels = [
    f"rows_{'_'.join(map(str, row))}_rep{i}"
    for row in rows_to_use
    for i in arr
]
repeated_rows = [
    row
    for row in rows_to_use
    for _ in range(n_repeats)
]

param_sets = list(zip(repeated_rows, labels))

# Run simulations in parallel
results = Parallel(n_jobs=base_config['number_of_parallel_parameters'])(
    delayed(process_param_set)(rows, label, base_config)
    for rows, label in tqdm(param_sets, total=len(param_sets), desc="All iterations")
)

# Print results
for (rows, label), res in zip(param_sets, results):
    print(f"✅ Completed simulation for {label} (rows={rows}): {res}")

[Worker rows_0_1_rep13] Using 12 threads for rows=[0, 1]

[Worker rows_0_1_rep12] Using 12 threads for rows=[0, 1]

{'{k_on_gene_1}': 0.66, '{k_off_gene_1}': 8.8, '{k_prod_mRNA_gene_1}': 2.0, '{k_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}': 2.0, '{k_add_gene_1_to_gene_2}': 6.0, '{n_gene_2_to_gene_1}': 2.0, '{k_add_gene_2_to_gene_1}': 6.0, '{n_gene_1_to_gene_3}': 2.0, '{k_add_gene_1_to_gene_3}': 6.0, '{n_gene_2_to_gene_3}': 2.0, '{k_add_gene_2_to_gene_3}': 6.0, '{pair_id_gene_1}': 0.0, '{gene_id_gene_1}': 1.0, '{k_deg_mRNA_gene_1}': 0.17328679513998632, '{k_deg_protein_gene_1}': 0.015403270679109895, '{k_on_gene_2}': 0.66, '{k_off_gene_2}': 8.8, '{k_prod_mRNA_gene_2}': 2.0, '{k_prod_protein_gene_2}': 560.0, '{pair_id_gene_2}': 0.0, '{gene_id_gene_2}': 2.0, '{k_deg_mRNA_gene_2}': 0.17328679513998632, '{k_deg_protein_gene_2}': 0.015403270679109895}
{'{k_on_gene_1}': 0.66, '{k_off_gene_1}': 8.8, '{k_prod_mRNA_gene_1}': 2.0, '{k_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}':

[Worker rows_0_1_rep14] Using 12 threads for rows=[0, 1]

{'{k_on_gene_1}': 0.66, '{k_off_gene_1}': 8.8, '{k_prod_mRNA_gene_1}': 2.0, '{k_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}': 2.0, '{k_add_gene_1_to_gene_2}': 6.0, '{n_gene_2_to_gene_1}': 2.0, '{k_add_gene_2_to_gene_1}': 6.0, '{n_gene_1_to_gene_3}': 2.0, '{k_add_gene_1_to_gene_3}': 6.0, '{n_gene_2_to_gene_3}': 2.0, '{k_add_gene_2_to_gene_3}': 6.0, '{pair_id_gene_1}': 0.0, '{gene_id_gene_1}': 1.0, '{k_deg_mRNA_gene_1}': 0.17328679513998632, '{k_deg_protein_gene_1}': 0.015403270679109895, '{k_on_gene_2}': 0.66, '{k_off_gene_2}': 8.8, '{k_prod_mRNA_gene_2}': 2.0, '{k_prod_protein_gene_2}': 560.0, '{pair_id_gene_2}': 0.0, '{gene_id_gene_2}': 2.0, '{k_deg_mRNA_gene_2}': 0.17328679513998632, '{k_deg_protein_gene_2}': 0.015403270679109895}
[Worker rows_0_1_rep15] Using 12 threads for rows=[0, 1]

{'{k_on_gene_1}': 0.66, '{k_off_gene_1}': 8.8, '{k_prod_mRNA_gene_1}': 2.0, '{k_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}':

[Worker rows_0_1_rep16] Using 12 threads for rows=[0, 1]

{'{k_on_gene_1}': 0.66, '{k_off_gene_1}': 8.8, '{k_prod_mRNA_gene_1}': 2.0, '{k_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}': 2.0, '{k_add_gene_1_to_gene_2}': 6.0, '{n_gene_2_to_gene_1}': 2.0, '{k_add_gene_2_to_gene_1}': 6.0, '{n_gene_1_to_gene_3}': 2.0, '{k_add_gene_1_to_gene_3}': 6.0, '{n_gene_2_to_gene_3}': 2.0, '{k_add_gene_2_to_gene_3}': 6.0, '{pair_id_gene_1}': 0.0, '{gene_id_gene_1}': 1.0, '{k_deg_mRNA_gene_1}': 0.17328679513998632, '{k_deg_protein_gene_1}': 0.015403270679109895, '{k_on_gene_2}': 0.66, '{k_off_gene_2}': 8.8, '{k_prod_mRNA_gene_2}': 2.0, '{k_prod_protein_gene_2}': 560.0, '{pair_id_gene_2}': 0.0, '{gene_id_gene_2}': 2.0, '{k_deg_mRNA_gene_2}': 0.17328679513998632, '{k_deg_protein_gene_2}': 0.015403270679109895}
[Worker rows_0_1_rep17] Using 12 threads for rows=[0, 1]

{'{k_on_gene_1}': 0.66, '{k_off_gene_1}': 8.8, '{k_prod_mRNA_gene_1}': 2.0, '{k_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}':

# Start running the simulation


In [5]:
os.makedirs(base_config['output_folder'], exist_ok=True)
rows_to_use = base_config['rows_to_use']
labels = ["rows_" + "_".join(map(str, row)) for row in rows_to_use]
param_sets = list(zip(rows_to_use, labels))

# ✅ Run over specified number of jobs (workers)
results = Parallel(n_jobs=base_config['number_of_parallel_parameters'])(
    delayed(process_param_set)(rows, label, base_config)
    for rows, label in tqdm(param_sets, total=len(param_sets), desc="All iterations")
)

# ✅ Print results (matching the concurrent.futures style)
for (rows, label), res in zip(param_sets, results):
    print(f"Completed simulation for {label} (rows={rows}): {res}")

[Worker rows_0_1_2] Using 8 threads for rows=[0, 1, 2]



,species1,change1,species2,change2,time,propensity
0,gene_1_A,1,gene_1_I,-1,-,{k_on_gene_1}*gene_1_I
1,gene_1_I,1,gene_1_A,-1,-,{k_off_gene_1}*gene_1_A
2,gene_1_mRNA,-1,-,-,-,{k_deg_mRNA_gene_1}*gene_1_mRNA
3,gene_1_mRNA,1,-,-,-,{k_prod_mRNA_gene_1}*gene_1_A
4,gene_1_protein,-1,-,-,-,{k_deg_protein_gene_1}*gene_1_protein
5,gene_1_protein,1,-,-,-,{k_prod_protein_gene_1}*gene_1_mRNA
6,gene_2_A,1,gene_2_I,-1,-,{k_on_gene_2}*gene_2_I + ((1*{k_add_gene_1_to_...
7,gene_2_I,1,gene_2_A,-1,-,{k_off_gene_2}*gene_2_A
8,gene_2_mRNA,-1,-,-,-,{k_deg_mRNA_gene_2}*gene_2_mRNA
9,gene_2_mRNA,1,-,-,-,{k_prod_mRNA_gene_2}*gene_2_A


All iterations:   0%|          | 0/1 [01:21<?, ?it/s]


ValueError: Parameter csv file not found at path: /Users/laj2116/Desktop/TWINFER/grnInference//data/example_simulation_input/median_parameter.csv